In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

## Part 1 — CoOp on Sketch-200, 3 seeds

### Prepare the dataset

In [ ]:
from imagenet_r_classes import r_class_names, r_wnids, wnid_to_r_index

In [ ]:
pip install datasets==2.16.0

In [ ]:
from datasets import load_dataset
dataset = load_dataset("songweig/imagenet_sketch")

In [ ]:
import json
from torchvision.datasets.utils import download_url

# Download the official ImageNet class index mapping
download_url(
    "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
    "./",
    "imagenet_class_index.json",
)

# Load the JSON mapping file
with open("./imagenet_class_index.json", "r") as f:
    class_idx = json.load(f)

# Convert to a list where index 0-999 corresponds to model outputs
class_names = [class_idx[str(i)][1] for i in range(1000)]

In [ ]:
wnid_to_idx = {class_idx[str(i)][0]: i for i in range(1000)}
idx_to_wnid = {i: class_idx[str(i)][0] for i in range(1000)}

In [ ]:
r_idx_1000 = [wnid_to_idx[w] for w in r_wnids]
idx_to_r_index = {wnid_to_idx[w]: i for w, i in wnid_to_r_index.items()}

In [ ]:
from datasets import ClassLabel

keep = set(r_idx_1000)
sk200 = dataset.filter(lambda y: y in keep, input_columns="label")

In [ ]:
new_features = sk200['train'].features.copy()
new_features["label"] = ClassLabel(names=r_class_names)

sk200 = sk200.map(
    lambda y: {"label": idx_to_r_index[y]},
    input_columns="label",
    features=new_features,
)

In [ ]:
type(sk200)

### Build the coop side

In [ ]:
from coop import PromptLearner, TextEncoderWrapper

In [ ]:
text_encoder = TextEncoderWrapper(model)

### Build the zero shot classifier

In [ ]:
from clip_zeroshot import load_cached_text_features, load_cached_image_features

In [ ]:
text_feature_cache = load_cached_text_features('/content/features/r_text-features.pt')
text_features = text_feature_cache['text_features']

### Prepare the features

In [ ]:
sk200_all_features = load_cached_image_features('./features/sk200_all_features.pt')

### Training loop

In [ ]:
import collections
import random

def split_indices(labels, seed, n_cache=16, n_val=10):
    rng = random.Random(seed)
    by_class = collections.defaultdict(list)
    for i, y in enumerate(labels):
        by_class[y].append(i)
    cache, val, test = [], [], []
    for idx in by_class.values():
        idx = idx[:]
        rng.shuffle(idx)
        cache += idx[:n_cache]
        val += idx[n_cache:n_cache + n_val]
        test += idx[n_cache + n_val:]
    return cache, val, test

In [ ]:
for param in model.parameters():
  param.requires_grad_(False)

In [ ]:
def train_coop(epochs, optimizer, num_ctx, logit_scale, train_loader):
  for epoch in range(epochs):
    total_loss = 0
    for img_feat, labels in tqdm(train_loader):
      img_feat = img_feat.to(device)
      labels = labels.to(device)

      prompts, tok_prompts = prompt_learner()
      text_features = text_encoder(prompts, tok_prompts)
      text_features = text_features / text_features.norm(dim=-1,keepdim=True)

      logits = logit_scale * img_feat @ text_features.t()
      loss = F.cross_entropy(logits, labels)
      total_loss += loss.item()

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    epoch_loss = total_loss / len(train_loader)
    print(f"epoch : {epoch + 1}, loss : {epoch_loss: .4f}")

In [ ]:
from torch.utils.data import TensorDataset
from tqdm.notebook import tqdm
import torch.nn.functional as F

epochs = 10
num_ctx = 4
logit_scale = model.logit_scale.exp()

labels_all = sk200['train']['label']

for sd in [42, 43, 44]:
  torch.manual_seed(sd)
  prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4,
                                   tokenizer=tokenizer, ctx_dim=512,
                                   class_names=r_class_names).to(device)
  optimizer = torch.optim.Adam([prompt_learner.ctx], lr=0.002)

  print(sum(p.numel() for p in prompt_learner.parameters()))

  cache_idx, val_idx, test_idx = split_indices(labels_all, seed=sd)

  few_shot_img_feats = sk200_all_features['image_features'][cache_idx]
  few_shot_labels = sk200_all_features['labels'][cache_idx]

  train_ds = TensorDataset(few_shot_img_feats, few_shot_labels)
  train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

  train_coop(epochs, optimizer, num_ctx, logit_scale, train_loader)

  print(f"Currently executing: seed {sd}")
  torch.save(prompt_learner.ctx.detach().cpu(), f"./features/coop_ctx_sk200_seed{sd}.pt")


### Evaluation Loop

In [ ]:
import pandas as pd
from harness import run_comparison, zero_shot_logits, coop_logits, accuracy, ece, signed_gap

metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}

feats = sk200_all_features["image_features"]
lbls = sk200_all_features["labels"]
labels_all = sk200["train"]["label"]

rows = []
for sd in [42, 43, 44]:
    cache_idx, val_idx, test_idx = split_indices(labels_all, seed=sd)

    pl = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer,
                       ctx_dim=512, class_names=r_class_names).to(device)
    pl.ctx.data.copy_(torch.load(f"./features/coop_ctx_sk200_seed{sd}.pt").to(device))

    with torch.no_grad():
        prompts, tok = pl()
        coop_text = text_encoder(prompts, tok)
        coop_text = coop_text / coop_text.norm(dim=-1, keepdim=True)

    shared = {
        "test_features": feats[test_idx].to(device),
        "labels": lbls[test_idx].to(device),
        "text_features": text_features.to(device),
        "coop_text_features": coop_text.to(device),
        "logit_scale": model.logit_scale.exp(),
    }
    methods = {
        "zero_shot": {"fn": zero_shot_logits, "params": {}},
        "coop":      {"fn": coop_logits,      "params": {}},
    }

    res = run_comparison(shared, methods, metrics)
    zs, co = res["zero_shot"], res["coop"]

    rows.append({
        "seed": sd, "n_test": len(test_idx),
        "zs_gap": zs["signed_gap"], "coop_gap": co["signed_gap"],
        "delta": co["signed_gap"] - zs["signed_gap"],
        "zs_acc": zs["accuracy"], "coop_acc": co["accuracy"],
        "zs_ece": zs["ece"], "coop_ece": co["ece"],
    })

coop_sk200 = pd.DataFrame(rows)
coop_sk200.round(2)

In [ ]:
s = coop_sk200.drop(columns=["seed", "n_test"]).agg(["mean", "min", "max"]).T
s["range"] = s["max"] - s["min"]
print(s.round(2))

coop_sk200.to_csv("./results/coop_sk200_seeds.csv", index=False)

In [ ]:
coop_sk200.round(2).to_string()

## Part 2 — TPT on Sketch-200

In [ ]:
from harness import run_tpt, accuracy, ece, signed_gap

In [ ]:
import collections
import random

def split_indices(labels, seed, n_cache=16, n_val=10):
    rng = random.Random(seed)
    by_class = collections.defaultdict(list)
    for i, y in enumerate(labels):
        by_class[y].append(i)
    cache, val, test = [], [], []
    for idx in by_class.values():
        idx = idx[:]
        rng.shuffle(idx)
        cache += idx[:n_cache]
        val += idx[n_cache:n_cache + n_val]
        test += idx[n_cache + n_val:]
    return cache, val, test

In [ ]:
import torchvision.transforms as transforms

augment_transform = transforms.Compose([
    transforms.Lambda(lambda im: im.convert("RGB")),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711)) # from clip
])

In [ ]:
_, _, test_idx = split_indices(sk200['train']['label'], seed=42)
rng = random.Random(42)
tpt_idx = rng.sample(test_idx, 500)
tpt_images = [sk200['train'][i]['image'] for i in tpt_idx]
tpt_labels = [sk200['train'][i]['label'] for i in tpt_idx]

In [ ]:
from coop import PromptLearner, TextEncoderWrapper

text_encoder = TextEncoderWrapper(model)
tpt_prompt_learner = PromptLearner(clip_model=model, device=device, n_ctx=4, tokenizer=tokenizer, ctx_dim=512, class_names=r_class_names).to(device)

In [ ]:
for p in model.parameters():
    p.requires_grad_(False)
print(sum(p.requires_grad for p in model.parameters()))

In [ ]:
metrics = {"accuracy": accuracy, "ece": ece, "signed_gap": signed_gap}
rows = []

for run_seed in [0, 1]:
    torch.manual_seed(run_seed)
    res = run_tpt(model, tpt_prompt_learner, text_encoder, preprocess,
                  tpt_images, tpt_labels, device, augment_transform, metrics, subset=500)
    rows.append({"run_seed": run_seed, **res})

In [ ]:
import pandas as pd

tpt_sk200 = pd.DataFrame(rows)

In [ ]:
tpt_sk200.to_csv("./results/tpt_sk200.csv", index=False)

In [ ]:
tpt_sk200.round(2)

#### Compare against zero shot clip

In [ ]:
from clip_zeroshot import build_and_cache_text_features, load_cached_image_features

In [ ]:
from harness import zero_shot_logits, run_comparison

In [ ]:
sk200_all_features = load_cached_image_features('/content/features/sk200_all_features.pt')

In [ ]:
single_template = ["a photo of a {}."]
st_cached = build_and_cache_text_features(model, tokenizer, r_class_names,
                                         single_template, device, "./features",
                                         "r_text_features_single")
st_text = st_cached

shared = {
    "test_features": sk200_all_features['image_features'][tpt_idx].to(device),
    "labels": sk200_all_features['labels'][tpt_idx].to(device),
    "text_features": st_text.to(device),
    "logit_scale": model.logit_scale.exp(),
}

methods = {"zero_shot_single": {"fn": zero_shot_logits, "params": {}}}
st_result = run_comparison(shared, methods, metrics)
st_result